<a href="https://colab.research.google.com/github/D2718281828nis/ML-MachineLearning-Graphs/blob/main/healthcare-gnn-lab/notebooks/01_patient_similarity_mi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MI patient-similarity graph

**Educational research code—not a diagnostic system.** UCI MI full-data adapter requires an explicitly selected complication after inspecting prevalence and prediction-time availability. The runnable smoke mode uses non-clinical synthetic patients.

This notebook intentionally delegates graph construction, splitting, training, and evaluation to tested package functions. It compares logistic regression, a Euclidean GCN, and a tangent-aggregation Poincaré variant under the same patient split. A clean Colab run has not yet been recorded; do not treat illustrative schemas as reproduced performance.

## Setup

Clone the current repository and install the package. Colab network access is required.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = "https://github.com/D2718281828nis/ML-MachineLearning-Graphs.git"
if Path("/content").exists():
    checkout = Path("/content/ML-MachineLearning-Graphs")
    if not checkout.exists():
        subprocess.run(["git", "clone", "-q", REPOSITORY, str(checkout)], check=True)
    lab_dir = checkout / "healthcare-gnn-lab"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(lab_dir)],
        check=True,
    )
else:
    candidates = [Path.cwd(), Path.cwd() / "healthcare-gnn-lab"]
    lab_dir = next(path for path in candidates if (path / "pyproject.toml").exists())

os.chdir(lab_dir)
src_dir = str(lab_dir / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
print(f"Healthcare lab: {lab_dir}")


Healthcare lab: /workspace/ML-MachineLearning-Graphs/healthcare-gnn-lab


None

## Run deterministic synthetic smoke experiment

The smoke fixture exists only to test leakage controls and the software path. It contains no real clinical records and supports no medical conclusion.

In [2]:
command = [
    sys.executable, "-m", "healthcare_gnn.cli",
    "--config", "configs/mi_smoke.json",
    "--output", "reports/mi_smoke.json",
]
environment = os.environ.copy()
environment["PYTHONPATH"] = src_dir
completed = subprocess.run(
    command, check=True, text=True, capture_output=True, env=environment
)
print(completed.stdout.strip())


{"output": "reports/mi_smoke.json", "status": "synthetic_smoke_not_clinical", "example": "mi"}


None

## Inspect structured output

Thresholds are selected on validation data. The report distinguishes patient split, graph diagnostics, edge-drop audit, model metrics, curvature, versions, and output schema.

In [3]:
import json
from pathlib import Path
report = json.loads(Path("reports/mi_smoke.json").read_text())
assert report["status"] == "synthetic_smoke_not_clinical"
report


{'schema_version': '1.0', 'status': 'synthetic_smoke_not_clinical', 'example': 'mi', 'seed': 17, 'config_snapshot': {'example': 'mi', 'mode': 'synthetic', 'seed': 17, 'n_samples': 180, 'n_features': 20, 'k': 3, 'epochs': 8, 'models': ['logistic', 'gcn', 'hyperbolic_gcn'], 'strict_inductive': True}, 'versions': {'python': '3.14.4', 'numpy': '2.5.2', 'sklearn': '1.9.0', 'torch': '2.13.0+cu130', 'torch_geometric': '2.8.0.post1'}, 'split': {'unit': 'patient', 'train': 108, 'validation': 36, 'test': 36}, 'graph': {'density': 0.027312228429546864, 'isolated_nodes': 0, 'connected_components': 1, 'mean_degree': 4.888888888888889, 'max_degree': 14, 'edge_homophily': 0.6977272727272728, 'label_mixing': 0.30227272727272725}, 'edge_drop_audit': {'density': 0.017535692116697702, 'isolated_nodes': 5, 'connected_components': 7, 'mean_degree': 3.138888888888889, 'max_degree': 11, 'edge_homophily': 0.6938053097345133, 'label_mixing': 0.30619469026548674}, 'models': [{'model': 'logistic', 'threshold_sel

## Interpretation contract

If the GNN or hyperbolic adaptation does not outperform logistic regression, report that result plainly. A lower training loss is not evidence that graph structure or curvature helps. Before a real-data run, follow the dataset-specific access, split, target, preprocessing, calibration, uncertainty, and subgroup checklist in the project README.